## 01. HyperOpt(Hyperparameter Optimization)

HyperOpt는 하이퍼파라미터 후보를 더 효율적으로 찾기 위한 최적화 도구이다.

하이퍼파라미터 튜닝에서는 다음과 같은 질문을 해결해야 한다.

- `C`는 0.1이 좋을지, 1이 좋을지 비교함.
- `max_depth`는 3이 좋을지, 5가 좋을지 비교함.
- `learning_rate`는 어느 정도가 적절한지 비교함.

GridSearchCV는 사람이 정한 후보 조합을 모두 확인한다.
후보가 적으면 이해하기 쉽지만, 후보가 많아지면 시간이 오래 걸린다.

RandomizedSearchCV는 후보 중 일부를 무작위로 확인한다.
GridSearchCV보다 빠를 수 있지만, 이전 실험에서 좋았던 방향을 적극적으로 활용하지는 않는다.

---

HyperOpt는 모델을 여러 번 학습해 보면서, 이전 trial 결과를 참고해 다음에 시도할 후보를 더 똑똑하게 고른다.
즉, 아무 후보나 계속 찍어보는 것이 아니라 성능이 좋아질 가능성이 높은 영역을 점점 더 집중적으로 탐색한다.

HyperOpt를 사용할 때는 보통 세 가지를 정의한다.

- 탐색 공간(search space): 어떤 하이퍼파라미터를 어떤 범위에서 찾을지 정함.
- 목적 함수(objective function): 후보 하나를 넣었을 때 모델을 학습하고 점수를 계산하는 함수임.
- trial 횟수: 후보를 몇 번 시도해 볼지 정함.

정리하면 HyperOpt는 다음과 같은 상황에서 사용한다.

- 후보 조합이 너무 많아 GridSearchCV가 오래 걸릴 때 사용함.
- 완전 무작위 탐색보다 더 효율적으로 좋은 후보를 찾고 싶을 때 사용함.
- 하이퍼파라미터 튜닝 과정을 최적화 문제처럼 다루고 싶을 때 사용함.

단, HyperOpt가 항상 최고의 모델을 보장하는 것은 아니다.
탐색 공간을 어떻게 정하는지, trial을 몇 번 수행하는지, 목적 함수를 어떻게 만드는지에 따라 결과가 달라질 수 있다.

**핵심 용어**
- search space: 탐색할 하이퍼파라미터 후보 범위.
- objective function: 후보를 받아 모델을 평가하고 손실값을 반환하는 함수.
- trial: 하나의 후보 조합으로 수행한 실험.
- loss: 최소화할 값. accuracy처럼 높을수록 좋은 지표는 음수로 바꿔 loss로 사용함.
- TPE: 이전 trial 결과를 참고해 다음 후보를 선택하는 HyperOpt의 대표 탐색 알고리즘.


## 02. 패키지 설치

HyperOpt는 scikit-learn 기본 패키지가 아니므로 설치가 필요하다.
이미 설치되어 있으면 아래 셀은 건너뛰어도 된다.

In [7]:
# %pip install hyperopt "setuptools==80.9.0"


Note: you may need to restart the kernel to use updated packages.


## 03. 실습 환경 준비

Wine 데이터와 RandomForestClassifier를 사용해 HyperOpt 기반 탐색 흐름을 확인한다.

In [8]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.metrics import accuracy_score, classification_report

from hyperopt import hp, fmin, tpe, Trials, STATUS_OK, space_eval


## 04. search space 정의

HyperOpt가 탐색할 하이퍼파라미터 후보 범위이다.
`hp.choice()`: 후보 목록 중 하나를 선택하는 search space를 만듦

In [9]:
# hp: 하이퍼 파라미터 탐색 공간을 정의할 때 사용하는 속성

# hp.choice(label, choices)
# - choices 목록 중 하나를 선택하는 탐색 공간을 만든다
search_space = {
    'n_estimators': hp.choice('n_estimators', [50, 100, 150]),
    'max_depth': hp.choice('max_depth', [2, 3, 4, 5, None]),
    'min_samples_leaf': hp.choice('min_samples_leaf', [1, 2, 4]),
}

## 05. Wine 데이터 분리

HyperOpt도 모델 선택 도구이므로 test 데이터는 마지막 확인용으로 남겨둔다.
objective 함수 안에서는 train 데이터에 대한 교차검증 점수만 사용한다.

In [10]:
wine = load_wine(as_frame=True)
hyper_X = wine.data
hyper_y = wine.target

hyper_X_train, hyper_X_test, hyper_y_train, hyper_y_test = train_test_split(
    hyper_X,
    hyper_y,
    test_size=0.2,
    random_state=42,
    stratify=hyper_y
)

print('train:', hyper_X_train.shape, hyper_y_train.shape)
print('test:', hyper_X_test.shape, hyper_y_test.shape)


train: (142, 13) (142,)
test: (36, 13) (36,)


## 06. objective 함수 정의

objective 함수는 하이퍼파라미터 후보를 받아 모델을 만들고, 검증 점수를 계산해 loss를 반환한다.
HyperOpt는 이 loss를 최소화하는 방향으로 다음 후보를 찾음.

In [11]:
# 매개 변수 params
# - HyperOpt가 search_space에서 뽑은 하이퍼파라미터 후보(dict type)

def objective(params):

    model = RandomForestClassifier(
        **params, # 전달 받은 dict 형태의 하이퍼파라미터 후보를 언팩킹
        random_state=42
    )

    # cross_val_score() :
    # train 데이터를 fold하여 교차 검증 후 fold별 점수 반환
    scores = cross_val_score(
        model,
        hyper_X_train,
        hyper_y_train,
        cv=5,
        scoring='accuracy'
    )

    # fold별 score 평균
    mean_score = scores.mean()

    # HyperOpt는 loss를 "최소화" 하는 도구
    # -> loss를 점점 작아지게 만들어야 한다
    # -> 높을수록 정확하다는 accuracy에 -1을 곱해서
    #    accuracy가 높을수록 loss가 작아지는 형태를 만듦.
    # ex) accuracy: 0.9        -> loss: -0.9
    #     accuracy: 0.95 (증가) -> loss: -0.95(감소)
    return {
        'loss': -mean_score,
        'mean_score': mean_score, # accuracy(정확도) -> 사람이 보는 용도
        'status': STATUS_OK, # 함수가 정상 실행 되었음을 의미
        'params': params  # 사용한 하이퍼 파라미터 후보
    }

In [12]:
# 정의한 objective 함수 확인
sample_result = objective({
    'n_estimators': 50,
    'max_depth': 3,
    'min_samples_leaf': 1,
})

print(sample_result)

{'loss': np.float64(-0.9862068965517242), 'mean_score': np.float64(0.9862068965517242), 'status': 'ok', 'params': {'n_estimators': 50, 'max_depth': 3, 'min_samples_leaf': 1}}


## 07. fmin으로 탐색 실행

`fmin()`: search space에서 후보를 고르고 objective 함수를 반복 실행함
`Trials`: 각 실험의 결과가 저장되는 객체

In [13]:

# fmin() : objective 함수의 loss가 작아지는 하이퍼파라미터 후보 탐색

trials = Trials() # HyperOpt가 수행한 실험 정보를 저장하는 객체

best_raw = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest, # TPE 알고리즘을 사용해서 다음 후보를 제안
    max_evals=20,  # 실험 횟수 지정(20번)
    trials= trials, # 실험 결과 저장
    rstate= np.random.default_rng(42)
)

# 1. search_space에 정의된 후보 범위에서 하이퍼파라미터 조합을 하나 고릅니다.
# 2. 그 후보를 objective() 함수에 넣습니다.
# 3. objective() 함수 안에서 모델을 만들고 교차검증 점수를 계산합니다.
# 4. 점수를 loss로 바꿔 HyperOpt에게 반환합니다.
# 5. HyperOpt는 loss가 작아지는 방향으로 다음 후보를 다시 고릅니다.
# 6. 이 과정을 max_evals=20이므로 총 20번 반복합니다.
# 7. 20번 실험 중 가장 좋은 후보를 best_raw에 저장합니다.

# space_eval() : 지정된 공간에서 raw index에 맞춰 하이퍼 파라미터를 반환
best_parms = space_eval(search_space, best_raw)

print('raw best:', best_raw)
print('best_params:', best_parms)

100%|██████████| 20/20 [00:14<00:00,  1.40trial/s, best loss: -0.9862068965517242]
raw best: {'max_depth': np.int64(3), 'min_samples_leaf': np.int64(2), 'n_estimators': np.int64(0)}
best_params: {'max_depth': 5, 'min_samples_leaf': 4, 'n_estimators': 50}


## 08. Trials 결과 확인

Trials에는 각 실험의 loss와 점수가 들어 있다.
결과를 DataFrame으로 바꾸면 어떤 후보들이 시도되었는지 확인하기 쉽다.

In [14]:
trial_rows = []

for trial_no, trial in enumerate(trials.trials, start=1):
    # trial['result']에는 objective 함수가 반환한 dict가 저장되어 있다.
    result = trial['result']

    # objective에서 직접 저장해 둔 params를 꺼낸다.
    params = result['params']

    trial_rows.append({
        'trial': trial_no,
        'loss': result['loss'],
        'mean_score': result['mean_score'],
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'min_samples_leaf': params['min_samples_leaf']
    })

# HyperOpt는 loss를 최소화하므로 표의 위쪽 후보가 더 좋은 후보이다.
trials_df = pd.DataFrame(trial_rows).sort_values('loss')

display(trials_df.round(4))

,trial,loss,mean_score,n_estimators,max_depth,min_samples_leaf
0,1,-0.9862,0.9862,50,5.0,4
1,2,-0.9862,0.9862,150,3.0,1
2,3,-0.9862,0.9862,50,3.0,1
4,5,-0.9862,0.9862,50,4.0,2
7,8,-0.9862,0.9862,100,5.0,1
5,6,-0.9862,0.9862,150,3.0,4
9,10,-0.9862,0.9862,150,4.0,1
8,9,-0.9862,0.9862,50,3.0,4
15,16,-0.9862,0.9862,50,4.0,4
13,14,-0.9862,0.9862,150,5.0,1


## 09. 최종 모델 학습과 test 평가

탐색이 끝나면 선택된 하이퍼파라미터로 모델을 다시 학습하고, 마지막으로 test 데이터에서 성능을 확인한다.

In [15]:
final_model = RandomForestClassifier(
    **best_parms,
    random_state=42
)

# 학습
final_model.fit(hyper_X_train, hyper_y_train)

# 예측
final_test_pred = final_model.predict(hyper_X_test)

# 평가
print("test_accuracy:", accuracy_score(hyper_y_test, final_test_pred))
print(classification_report(hyper_y_test, final_test_pred))

test_accuracy: 0.9722222222222222
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        12
           1       1.00      0.93      0.96        14
           2       1.00      1.00      1.00        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36



## 10. 정리

- HyperOpt는 하이퍼파라미터 자동 탐색을 위한 선택 심화 도구임.
- search space는 탐색할 후보 범위이고, objective 함수는 후보를 평가해 loss를 반환함.
- accuracy처럼 높을수록 좋은 지표는 `-accuracy`로 바꿔 loss로 사용할 수 있음.
- 테스트 데이터는 HyperOpt 탐색에 사용하지 않고 마지막 확인용으로 남겨야 함.
- 수업 시간이 부족하면 개념과 `objective -> fmin -> best_params` 흐름만 소개해도 충분함.